In [1]:

# clone repo

import os

PROJECT_ROOT = "/content/Project_Generative_AI_for_Data_Augmentation"

if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/Jorj91/Project_Generative_AI_for_Data_Augmentation.git {PROJECT_ROOT}

%cd {PROJECT_ROOT}

Cloning into '/content/Project_Generative_AI_for_Data_Augmentation'...
remote: Enumerating objects: 290, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 290 (delta 55), reused 53 (delta 29), pack-reused 190 (from 1)
Receiving objects: 100% (290/290), 8.42 MiB | 18.87 MiB/s, done.
Resolving deltas: 100% (146/146), done.
/content/Project_Generative_AI_for_Data_Augmentation


In [2]:
# =============================
# CONTROLLED VERBOSITY
# =============================

# Disable HF download progress bars BEFORE importing anything HF-related

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

In [3]:
# Dependency install
INSTALL_DEPS = True

if INSTALL_DEPS:
    !pip install -r requirements.txt -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.3 MB/s eta 0:00:00


In [4]:
import sys
sys.path.append(os.path.join(PROJECT_ROOT, "src"))

import importlib
import captioning
importlib.reload(captioning)
from captioning import run_captioning

In [5]:
# Setup

import torch
import logging
from torchvision.datasets import OxfordIIITPet
import numpy as np
from torch.utils.data import Subset
from sklearn.model_selection import train_test_split
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import random
from transformers.utils import logging as transformers_logging
from huggingface_hub.utils import logging as hf_logging


# Silence transformers & HF logs (keep only errors)
transformers_logging.set_verbosity_error()
hf_logging.set_verbosity_error()

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

In [6]:
# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [7]:
import gc

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print("GPU memory cleared.")

In [8]:
PROJECT_ROOT # it must be /content/Project_Generative_AI_for_Data_Augmentation

'/content/Project_Generative_AI_for_Data_Augmentation'

In [9]:
# control flags. REMEMBER TO USE THE OTHERS AS WELL IN THE NB!!!
RUN_CAPTIONING = False
RUN_TEXT_VARIATION = False
RUN_IMAGE_GENERATION = False
RUN_TRAINING = False

In [10]:
# dataset loading

dataset_train = OxfordIIITPet(
    root=os.path.join(PROJECT_ROOT, "data", "raw"),
    split="trainval",
    download=True
)

dataset_test = OxfordIIITPet(
    root=os.path.join(PROJECT_ROOT, "data", "raw"),
    split="test",
    download=True
)

print("Train size:", len(dataset_train))
print("Test size:", len(dataset_test))

100%|██████████| 792M/792M [05:24<00:00, 2.44MB/s]
100%|██████████| 19.2M/19.2M [00:05<00:00, 3.83MB/s]


Train size: 3680
Test size: 3669


In [11]:
# extract labels
labels = dataset_train._labels
indices = np.arange(len(dataset_train))

# perform stratified split
train_small_idx, _ = train_test_split(
    indices,
    train_size=0.30,
    stratify=labels,
    random_state=42
)

SPLIT_DIR = os.path.join(PROJECT_ROOT, "data", "splits")
os.makedirs(SPLIT_DIR, exist_ok=True)

np.save(os.path.join(SPLIT_DIR, "train_small_indices.npy"), train_small_idx)

In [12]:
# # run for ALL
# # create subset dataset from training set
dataset_train_small = Subset(dataset_train, train_small_idx)

In [13]:
# run for 10
dataset_train_small_10 = Subset(dataset_train, train_small_idx[:10])

# Captioning

In [14]:
CAPTION_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "captions",
    "captions_train_small_10.json"
)

In [15]:
if RUN_CAPTIONING:

  device = "cuda" if torch.cuda.is_available() else "cpu"

  processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")

  model = Blip2ForConditionalGeneration.from_pretrained(
      "Salesforce/blip2-opt-2.7b",
      torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32 # to reduce GPU memory usage
  )

  model.to(device)
  model.eval()



  captions_dict = run_captioning(
      # dataset_train_small=dataset_train_small, # FOR ALL
      dataset_train_small=dataset_train_small_10, # FOR 10
      model=model,
      processor=processor,
      device=device,
      output_path=CAPTION_PATH
  )

  del model
  del processor
  clear_gpu()
  !nvidia-smi

In [ ]:
'''
import nbformat
import os

def hard_clean_notebook(path):
    nb = nbformat.read(path, as_version=4)

    if "widgets" in nb.metadata:
        del nb.metadata["widgets"]

    for cell in nb.cells:
        if "widgets" in cell.get("metadata", {}):
            del cell["metadata"]["widgets"]

    nbformat.write(nb, path)
    print(f"Cleaned: {os.path.basename(path)}")


# 🔥 Walk entire project and clean every notebook
for root, _, files in os.walk(PROJECT_ROOT):
    for file in files:
        if file.endswith(".ipynb"):
            hard_clean_notebook(os.path.join(root, file))

print("All notebooks in project HARD cleaned.")
'''

# Text variation

## FLAN-T5-Large Model



In [16]:
import text_variation_flan_large
importlib.reload(text_variation_flan_large)
from text_variation_flan_large import run_text_variation as run_flan_large

In [17]:
MAX_ITEMS = 10

In [18]:
CAPTION_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "captions",
    "captions_train_small_10.json"
)

In [19]:
run_flan_large(
    caption_file=CAPTION_PATH,
    max_items=MAX_ITEMS
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
  0%|          | 0/10 [00:00<?, ?it/s]


Original: a pomeranian dog my dog is sitting on the bed
Generated: ['three-leggi', 'little german male standing dog by your apartment room bedroom balcony enjoying nature walks through some countryside as part of its annual activity on', 'Two pets lying naked. Some lying off leh side and are going towards. Three pictures then there with our pomelized German dogs one, two with two different dogs']


 10%|█         | 1/10 [00:04<00:36,  4.02s/it]


Original: a pomeranian dog this dog is sitting on the bed
Generated: ["An inefably adorable canter that wants his place, not gets anywhere where her new human loves at their pet cat peaked chair under bedrail that is right at door when puppy' the", 'the brown and the red puppies attracted all attention for months so decided we want our young dog the PooMd with collar as they all thought about who stole something like we just were.he', 'two of the dogs sleep and they play for 0 degrees to each dm in range the others sit at all types.a small female baby sitting up, in close close']

Original: a havanese dog is sitting on a tennis court
Generated: ['portrait view shows oax playing double fault football off on tennis with dogs under two feet deep underneath the players is doing their sports playing back field,the top four', 'It can sit in or walk - or standing for it?! And with those legs', 'an exotic looking horse or cow looks toward you by it court.eg black cow standing by by dog holdin

 20%|██        | 2/10 [00:06<00:25,  3.23s/it]

...


Original: a keeshond dog is standing on the grass
Generated: ['another black leander on dog leads down sidewalk after rain for his last round walk as one close dog pull down to play inside and bark out', 'He takes walks together with everyone near us playing sports all afternoon through several sunny skies on his way inside or by bus while rest', 'Two giant white fur cow standing along some lush, blue vista next as this pup has some snacks before doing sit up dog trick around side camera is doing chase to keep out pregabitat']

Original: a chihuahua dog is a small dog with a long body and short legs
Generated: ['with their round skull there always plenty enough elbow width evenness. as the body develop. A short body', 'The peconosa is usually one length bigger in span and two sets closer the mid pelvi, giving more muscle for body power and larger size head area relative at least half-way', 'large to skinny it goes fast enough and quick without pushing out excess strength its size with

 70%|███████   | 7/10 [00:16<00:06,  2.13s/it]


Original: a chihuahua dog is a small dog with a short body and a long tail
Generated: ['indian people own Chik indian-style poo which teaches obedience', 'with good ears long body it weigh moderate over 245c for puppies small but furly tail its ideal and is popular here american manufacturers have put very thin collar they put little money so very hot is', 'that dog had brown feet is often thought by animals of ,']


 70%|███████   | 7/10 [00:17<00:07,  2.46s/it]
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



KeyboardInterrupt: 

In [ ]:
clear_gpu()
!nvidia-smi

GPU memory cleared.
Fri Feb 20 11:57:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             52W /  400W |     548MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------------

FLAN-T5-Large was evaluated as a candidate model for caption rewriting.

However, qualitative analysis showed several consistent issues:

- **poor semantic preservation:** generated output often drifted away from original meaning

- **hallucinations:** frequent introduction of unrelated object, people, or scenes

Due to these limitations, FLAN-T5-Large was deemed unsuitable for controlled data augmentation.

## FLAN-T5-XL Model

In [ ]:
from text_variation_flan_xl import run_text_variation as run_flan_xl

run_flan_xl(
    caption_file=CAPTION_PATH,
    max_items=10
)

  0%|          | 0/10 [00:00<?, ?it/s]


Original: a pomeranian dog this dog is sitting on the bed
Generated: ['a pomeranian dog this pomeranian is sitting on the bed', 'a pomeranian dog this pomeranian dog is sitting on the bed', 'a pomeranian dog this pomeranian is sitting on the bed .']


 10%|█         | 1/10 [00:01<00:16,  1.89s/it]


Original: a pomeranian dog my dog is sitting on the bed
Generated: ['a pomeranian dog my pomeranian is sitting on the bed', 'a pomeranian dog my dog is sitting on the bed', 'a pomeranian dog my pomeranian dog is sitting on the bed']

Original: a havanese dog is sitting on the tennis court
Generated: ['A Havanese dog is sitting on a tennis court.', 'A Havanese dog sits on a tennis court.', 'A Havanese dog is sitting on the tennis court.']


 20%|██        | 2/10 [00:03<00:13,  1.67s/it]

...


Original: a saint bernard dog is standing in the snow
Generated: ['a saint bernard dog is standing in the snow', 'a saint bernard is standing in the snow', 'a saint bernard dog standing in the snow']

Original: a havanese dog is standing on a wooden staircase
Generated: ['A Havanese dog is standing on a wooden staircase.', 'A Havanese dog is standing on a wooden staircase', 'A Havanese is standing on a wooden staircase.']


 90%|█████████ | 9/10 [00:13<00:01,  1.38s/it]


Original: a havanese dog is sitting on the steps
Generated: ['A Havanese dog is sitting on the steps.', 'A Havanese dog sits on the steps.', 'A Havanese dog is sitting on the steps']

Original: a persian cat is looking angry
Generated: ['a persian cat looks angry', 'a persian cat is looking angry', 'a persian cat is angry']


100%|██████████| 10/10 [00:14<00:00,  1.48s/it]


Original: a persian cat is looking at the camera with an angry expression
Generated: ['a persian cat is looking at the camera with an angry expression', 'a persian cat is looking at the camera with an angry expression .', 'a persian cat is looking at the camera with an angry expression.']


In [ ]:
clear_gpu()
!nvidia-smi

GPU memory cleared.
Fri Feb 20 11:58:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             52W /  400W |     548MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------------

FLAN-T5-XL produced grammatically correct and semantically faithful rewrites.

However, the generated variations showed very low lexical diversity, often resulting in near-duplicate sentences with only minor wording or punctuation changes.

Since the goal of this stage is meaningful data augmentation, higher variation diversity was required.

## Mistral 7B Instruct Model

In [20]:
from text_variation_mistral import run_text_variation as run_mistral

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
CAPTIONS_DIR = os.path.join(DATA_DIR, "captions")
TEXT_VARIATIONS_DIR = os.path.join(DATA_DIR, "text_variations")


os.makedirs(TEXT_VARIATIONS_DIR, exist_ok=True)

CAPTION_FILE = os.path.join(
    CAPTIONS_DIR,
    "captions_train_small_10.json"
)

TEXT_VARIATION_FILE = os.path.join(
    TEXT_VARIATIONS_DIR,
    "text_variations_train_small_10.json"
)

run_mistral(
    caption_file = CAPTION_FILE,
    output_file= TEXT_VARIATION_FILE,
    max_items=10
)

100%|██████████| 10/10 [00:44<00:00,  4.46s/it]

Mistral text variations saved.


In [21]:
clear_gpu()
!nvidia-smi

GPU memory cleared.
Fri Feb 20 17:06:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             50W /  400W |    3706MiB /  40960MiB |      2%      Default |
|                                         |                        |             Disabled |
+---------------------------

# Caption Selection

In [22]:
from src.image_generation import (CaptionSelector, SyntheticImageGenerator)
import json

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [23]:
with open(TEXT_VARIATION_FILE, "r") as f:
    text_variations = json.load(f)

In [24]:
selector = CaptionSelector()

selected_data = {}

for idx, data in text_variations.items():

    class_name = data["class_name"]
    original_captions = data["original_captions"]
    generated_captions = data["generated_captions"]

    selected_generated = selector.select_top_captions(
        original_captions,
        generated_captions,
        top_k=2
    )

    selected_data[idx] = {
    "class_name": class_name,
    "original_captions": original_captions,
    "selected_generated_captions": selected_generated
}

In [25]:
selected_data

{'3037': {'class_name': 'Pomeranian',
  'original_captions': ['a pomeranian dog my dog is sitting on the bed',
   'a pomeranian dog this dog is sitting on the bed'],
  'selected_generated_captions': ['The Pomeranian breed of dog is occupying the bed where I am sitting.',
   'My Pomeranian dog is seated on the bed.']},
 '2635': {'class_name': 'Havanese',
  'original_captions': ['a havanese dog is sitting on a tennis court',
   'a havanese dog is sitting on the tennis court'],
  'selected_generated_captions': ['A Havanese dog is positioned on a tennis court.',
   'The Havanese dog is situated on a tennis court.']},
 '498': {'class_name': 'British Shorthair',
  'original_captions': ['a british shorthair cat is sitting in a box',
   'a british shorthair cat is sitting in a cardboard box'],
  'selected_generated_captions': ['In a box sits a British Shorthair cat.',
   'A British Shorthair cat is positioned inside a box.']},
 '1449': {'class_name': 'Samoyed',
  'original_captions': ['a samoy

# Image Generation

In [26]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())

CUDA available: True
Device count: 1


In [27]:
generator = SyntheticImageGenerator()

metadata = generator.generate_images(
            selected_data = selected_data,
            output_dir = os.path.join(DATA_DIR, "synthetic/images"),
            checkpoint_file = os.path.join(DATA_DIR, "synthetic/generation_checkpoint.json"),
            batch_size=4
    )

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

 10%|█         | 1/10 [00:03<00:27,  3.02s/it]

  0%|          | 0/20 [00:00<?, ?it/s]

...

 80%|████████  | 8/10 [00:16<00:03,  1.99s/it]

  0%|          | 0/20 [00:00<?, ?it/s]

 90%|█████████ | 9/10 [00:18<00:01,  1.99s/it]

  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:20<00:00,  2.07s/it]


In [28]:
import shutil

shutil.make_archive(
    "synthetic_images",
    'zip',
    os.path.join(DATA_DIR, "synthetic/images")
)

'/content/Project_Generative_AI_for_Data_Augmentation/synthetic_images.zip'

In [29]:
from google.colab import files
files.download("synthetic_images.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>